# 03 · Transfer learning con MobileNetV2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EstebanGuerrero125/steel-defect-detection-cnn/blob/main/notebooks/03_mobilenetv2_transfer.ipynb)

**Objetivo:** entrenar MobileNetV2 preentrenada en ImageNet y compararla contra la CNN baseline del notebook 02 (99,4 % de accuracy en test).

> En Colab activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

**Estrategia en dos fases:**

1. **Extracción de características:** la base MobileNetV2 queda congelada y solo se entrena la capa de clasificación nueva (6 clases).
2. **Ajuste fino (*fine-tuning*):** se descongelan las últimas 30 capas de la base y se entrenan con una tasa de aprendizaje muy baja (1e-5) para adaptarlas a las imágenes de acero.

**Diferencias con el baseline:**

| | CNN baseline | MobileNetV2 |
|---|---|---|
| Entrada | 200×200, 1 canal | 224×224, 3 canales (el gris se repite) |
| Normalización | [0, 1] | [-1, 1] |
| Pesos iniciales | Aleatorios | ImageNet |
| BatchNormalization | Se entrena (`momentum=0.9`) | Congelada en modo inferencia |

Las particiones (train / val / test) y el aumento de datos son **los mismos** del notebook 02, para que la comparación sea justa.

## 1. Configuración

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EstebanGuerrero125/steel-defect-detection-cnn.git"

if IN_COLAB:
    ROOT = Path("/content/steel-defect-detection-cnn")
    if ROOT.exists():
        # La carpeta sobrevive a "Reiniciar sesión": se actualiza para usar el código más reciente
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

commit = subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True).stdout.strip()
print("Código del repositorio en el commit:", commit)

sys.path.insert(0, str(ROOT))
# Si la celda se vuelve a ejecutar sin reiniciar, se descartan los módulos de src/ ya importados
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

FIG_DIR = ROOT / "reports" / "figures"
MODEL_DIR = ROOT / "models"

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

from src.dataset import CLASSES, add_metadata, build_index
from src.models import build_mobilenetv2, unfreeze_top_layers
from src.pipeline import SEED, make_dataset, make_splits

keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "no disponible")

## 2. Datos

In [ ]:
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

DATA_PATH = Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
df = add_metadata(build_index(DATA_PATH))
train_df, val_df, test_df = make_splits(df)
print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_ds = make_dataset(train_df, IMAGE_SIZE, channels=3, batch_size=BATCH_SIZE, shuffle=True)
val_ds = make_dataset(val_df, IMAGE_SIZE, channels=3, batch_size=BATCH_SIZE)
test_ds = make_dataset(test_df, IMAGE_SIZE, channels=3, batch_size=BATCH_SIZE)

## 3. Modelo

In [ ]:
def count_params(model):
    trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    return trainable, model.count_params()

model = build_mobilenetv2(IMAGE_SIZE)
trainable, total = count_params(model)
print(f"Parámetros totales: {total:,} | entrenables en la fase 1: {trainable:,}")
model.summary()

## 4. Fase 1: extracción de características

Solo se entrena la capa de clasificación (unos 7.700 parámetros). La base actúa como un extractor de características fijo.

In [ ]:
EPOCHS_HEAD = 20
EPOCHS_FINE = 30
MODEL_PATH = MODEL_DIR / "mobilenetv2_transfer.keras"

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

start = time.time()
history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
)
minutes_head = (time.time() - start) / 60

head_val_loss, head_val_acc = model.evaluate(val_ds, verbose=0)
model.save(MODEL_PATH)
print(f"Fase 1: {minutes_head:.1f} min | val_accuracy = {head_val_acc:.4f} | val_loss = {head_val_loss:.4f}")

## 5. Fase 2: ajuste fino

Se descongelan las últimas 30 capas de la base. La tasa de aprendizaje es 100 veces menor que en la fase 1 para no destruir lo aprendido en ImageNet.

Las capas **BatchNormalization se mantienen congeladas**. En una primera versión quedaban entrenables y, al empezar esta fase, la accuracy de train caía de 99,8 % a 68 %: pasaban a normalizar con las estadísticas de cada lote en lugar de las de ImageNet.

El modelo guardado solo se reemplaza si el ajuste fino **mejora** la pérdida de validación de la fase 1.

In [ ]:
N_UNFROZEN = 30
n_base_layers = unfreeze_top_layers(model, N_UNFROZEN)
trainable, total = count_params(model)
print(f"Capas descongeladas: {N_UNFROZEN} de {n_base_layers} | parámetros entrenables: {trainable:,} de {total:,}")

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

epochs_head = len(history_head.history["loss"])
start = time.time()
history_fine = model.fit(
    train_ds, validation_data=val_ds,
    initial_epoch=epochs_head, epochs=epochs_head + EPOCHS_FINE,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True,
                                        initial_value_threshold=head_val_loss),
    ],
)
minutes_fine = (time.time() - start) / 60
print(f"Fase 2: {minutes_fine:.1f} min, {len(history_fine.history['loss'])} épocas")

In [ ]:
hist = pd.concat([
    pd.DataFrame(history_head.history).assign(fase="1_extraccion"),
    pd.DataFrame(history_fine.history).assign(fase="2_ajuste_fino"),
], ignore_index=True)
hist.index = hist.index + 1
hist.index.name = "epoca"
hist.to_csv(ROOT / "reports" / "03_historial_mobilenetv2.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist[["loss", "val_loss"]].plot(ax=axes[0], title="Pérdida")
hist[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")
for ax, ylabel in zip(axes, ["Entropía cruzada", "Accuracy"]):
    ax.axvline(epochs_head + 0.5, color="gray", linestyle="--")
    ax.set(xlabel="Época", ylabel=ylabel)
    ax.legend(["train", "val", "inicio del ajuste fino"])
plt.tight_layout()
plt.savefig(FIG_DIR / "03_curvas_entrenamiento.png", dpi=150)
plt.show()

## 6. Evaluación en test

In [ ]:
best = keras.models.load_model(MODEL_PATH)
val_loss, val_acc = best.evaluate(val_ds, verbose=0)
test_loss, test_acc = best.evaluate(test_ds, verbose=0)

metrics = {
    "modelo": "mobilenetv2_transfer",
    "parametros": int(best.count_params()),
    "epocas_entrenadas": int(len(hist)),
    "epocas_fase_1": int(epochs_head),
    "epocas_fase_2": int(len(hist) - epochs_head),
    "minutos_entrenamiento": round(minutes_head + minutes_fine, 1),
    "val_accuracy_fase_1": round(float(head_val_acc), 4),
    "val_accuracy": round(float(val_acc), 4),
    "test_accuracy": round(float(test_acc), 4),
    "test_loss": round(float(test_loss), 4),
}
(ROOT / "reports" / "metrics_mobilenetv2.json").write_text(json.dumps(metrics, indent=2))
pd.Series(metrics)

## 7. Comparativa con la CNN baseline

Se comparan los dos modelos en cinco dimensiones:

- **Calidad:** accuracy en test (mismas 360 imágenes).
- **Velocidad de aprendizaje:** en qué época alcanza por primera vez 95 % de accuracy en validación.
- **Costo de entrenamiento:** épocas y minutos.
- **Tamaño:** parámetros y peso del archivo `.keras`.
- **Velocidad de inferencia:** milisegundos por imagen (relevante para la API y para inspección en línea).

In [ ]:
def epochs_to_reach(history_df, threshold=0.95):
    reached = history_df.index[history_df["val_accuracy"] >= threshold]
    return int(reached[0]) if len(reached) else None


def inference_ms(model, dataset, n_images, repeats=3):
    model.predict(dataset, verbose=0)  # calentamiento: la primera llamada compila el grafo
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        model.predict(dataset, verbose=0)
        times.append(time.perf_counter() - start)
    return min(times) / n_images * 1000


baseline_path = MODEL_DIR / "cnn_baseline.keras"
baseline = keras.models.load_model(baseline_path)
baseline_metrics = json.loads((ROOT / "reports" / "metrics_cnn_baseline.json").read_text())
baseline_hist = pd.read_csv(ROOT / "reports" / "02_historial_cnn_baseline.csv", index_col="epoca")
baseline_test_ds = make_dataset(test_df, 200, channels=1, batch_size=BATCH_SIZE)

comparison = pd.DataFrame({
    "CNN baseline": {
        "Accuracy test": baseline_metrics["test_accuracy"],
        "Errores en test (de 360)": round((1 - baseline_metrics["test_accuracy"]) * len(test_df)),
        "Accuracy val": baseline_metrics["val_accuracy"],
        "Época en que llega a 95 % val": epochs_to_reach(baseline_hist),
        "Épocas entrenadas": baseline_metrics["epocas_entrenadas"],
        "Minutos de entrenamiento": baseline_metrics["minutos_entrenamiento"],
        "Parámetros": baseline_metrics["parametros"],
        "Tamaño del archivo (MB)": round(baseline_path.stat().st_size / 1e6, 1),
        "Inferencia (ms por imagen)": round(inference_ms(baseline, baseline_test_ds, len(test_df)), 2),
    },
    "MobileNetV2": {
        "Accuracy test": metrics["test_accuracy"],
        "Errores en test (de 360)": round((1 - metrics["test_accuracy"]) * len(test_df)),
        "Accuracy val": metrics["val_accuracy"],
        "Época en que llega a 95 % val": epochs_to_reach(hist),
        "Épocas entrenadas": metrics["epocas_entrenadas"],
        "Minutos de entrenamiento": metrics["minutos_entrenamiento"],
        "Parámetros": metrics["parametros"],
        "Tamaño del archivo (MB)": round(MODEL_PATH.stat().st_size / 1e6, 1),
        "Inferencia (ms por imagen)": round(inference_ms(best, test_ds, len(test_df)), 2),
    },
}, dtype=object)
comparison.index.name = "metrica"
comparison.to_csv(ROOT / "reports" / "03_comparativa_modelos.csv")
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(baseline_hist.index, baseline_hist["val_accuracy"], label="CNN baseline (desde cero)")
ax.plot(hist.index, hist["val_accuracy"], label="MobileNetV2 (transfer learning)")
ax.axhline(0.95, color="gray", linestyle=":", label="95 %")
ax.set(title="Accuracy de validación por época", xlabel="Época", ylabel="Accuracy", ylim=(0, 1.02))
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_comparativa_val_accuracy.png", dpi=150)
plt.show()

## 8. Guardar resultados

Descarga un `.zip` con el modelo, las métricas, la comparativa y las figuras. Descomprímelo en la raíz del repositorio local.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    bundle = Path("/content/resultados_03")
    for rel in ["models/mobilenetv2_transfer.keras", "reports/metrics_mobilenetv2.json",
                "reports/03_historial_mobilenetv2.csv", "reports/03_comparativa_modelos.csv",
                "reports/figures/03_curvas_entrenamiento.png",
                "reports/figures/03_comparativa_val_accuracy.png"]:
        (bundle / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(ROOT / rel, bundle / rel)
    files.download(shutil.make_archive("/content/resultados_03", "zip", bundle))

## 9. Conclusiones

> Completa esta sección con tus resultados.

- **Calidad:** _¿qué modelo tiene mejor accuracy en test? ¿La diferencia es significativa con 360 imágenes?_
- **Velocidad de aprendizaje:** _¿cuántas épocas necesitó cada modelo para llegar a 95 % en validación?_
- **Aporte del ajuste fino:** _¿mejoró la fase 2 respecto a la fase 1?_
- **Costo:** _tamaño del modelo y tiempo de inferencia de cada uno._
- **Modelo elegido para la API:** _¿cuál y por qué?_

**Siguiente paso:** matriz de confusión y `classification_report` de ambos modelos (`04_evaluacion.ipynb`).